In [1]:
# Benjamin_ExtractGraves.py
# 4/25/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time0,listings_df
    if call_count%100==0:
        sleep_time =sleep_time0
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
        # listings_df.to_csv(fileOut,mode="w", index=False,header=True)
        # listings_df.to_csv(fileOut0,mode="w", index=False,header=True) 
    else:
        sleep_time = random.uniform(2, 4)
    return sleep_time 

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0


start_time=datetime.datetime.now()
event_time=start_time


process_name='Extract Benjamin Graves '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"


fileIn="Benjamin_Cemeteries"
fileIn="Benjamin_Cemeteries_Try"
fileOutput="Benjamin_Graves_Try"

if append==1:fileInputEvents='Benjamin_Cemeteries_X'


fileCemeteriesIn=DirMain+"OutputData\\Cemeteries\\"+fileIn+".csv"
fileOut=DirMain+"OutputData\\Graves\\"+fileOutput+"_"+timestamp+".csv"
fileOut0=DirMain+"OutputData\\Graves\\"+fileOutput+".csv"

cemetery_frame_df_header=['cemetery_id','memorial_count','cemetery','city','county','state','state_id','county_id']
cemetery_info_df=pd.read_csv(fileCemeteriesIn)
cemetery_info_df.drop(cemetery_info_df[cemetery_info_df["memorial_count"]<200].index, inplace=True) 

cemetery_id_list=cemetery_info_df["cemetery_id"].fillna(0).astype(int)
memorial_count_list=cemetery_info_df["memorial_count"].fillna(0).astype(int)
cemetery_list=cemetery_info_df["cemetery"]
city_list=cemetery_info_df["city"]
county_list=cemetery_info_df["county"]
state_list=cemetery_info_df["state"]
county_id_list=cemetery_info_df["county_id"]
state_id_list=cemetery_info_df["state_id"]

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}

if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

sleep_time0=2*60

url0 = "https://www.findagrave.com/cemetery/"
url1 ="/memorial-search?ajax=true&skip="
url2 ="&limit=100&firstname=&middlename=&lastname=&birthyear=&birthyearfilter=&deathyear=&deathyearfilter=&locationId=&location=&memorialid=&datefilter=&orderby=r&photofilter=&gpsfilter=&famous=&cenotaph=&monument=&isVeteran=&flowers=&sponsored=&noCemetery=&includeNickName=&includeMaidenName=&includeTitles=&hasPlot=&plot=&bio=&partialLastName=&exactName=&linkedToName=&fuzzyNames=&mcid="


# total_cemeteries=get_total_cemeteries(url0)

cemetery_count=-1
total_frame_count=0
total_cemetery_count=len(cemetery_id_list)
for cemetery_id in cemetery_id_list:
    memorial_count_list
    cemetery_count+=1
    memorial_count=memorial_count_list.iloc[cemetery_count]
    cemetery_name=cemetery_list.iloc[cemetery_count]
    cemetery_city=city_list.iloc[cemetery_count]
    cemetery_county=county_list.iloc[cemetery_count]
    cemetery_state=state_list.iloc[cemetery_count]

    frame_count=0
    grave_frame_count=0
    for frame in range(0,memorial_count+1,100):
        total_frame_count+=1
        frame_count+=1
        url=url0+str(cemetery_id)+url1+str(frame)+url2

        response = requests.request("GET", url, headers=my_header) 
        status_code=response.status_code
        url_source = response.text 

        if status_code!=200:print('\tstatus_code:',status_code, 'frame count:',frame_count);break
        beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
        specific_tag_text = beautifulSoupText.get_text()


        capture_json=json.loads(specific_tag_text) 
        grave_count=capture_json['result']['count']
        collection_df=pd.json_normalize(capture_json['collection'])
        try:
            collection_df['maidenName']   
        except:
            collection_df['maidenName']=np.nan
            
        try:
            collection_df['birthCountryName']
        except:
            collection_df['birthCountryName']=np.nan   

        try:
            collection_df['Parents']
        except:
            collection_df['Parents']=np.nan   

        try:
            collection_df['Siblings']
        except:
            collection_df['Siblings']=np.nan   

        try:
            collection_df['lastName']
        except:
            collection_df['lastName']=np.nan   
            

        collection_df['memorial_count']=memorial_count
        
        try:
            collection_df['cemeteryId']
        except:
             break
        grave_frame_df=pd.concat([collection_df['cemeteryId'],collection_df['cemeteryName'],collection_df['memorial_count'],collection_df['fullName'],collection_df['lastName'], collection_df['firstName'],\
                                      collection_df['maidenName'],collection_df['birthCountryName'],collection_df['birthDate'],collection_df['deathDate'],collection_df['Parents'],\
                                      collection_df['Siblings']], axis=1)   
        graves_df = graves_df._append(grave_frame_df, ignore_index = True) 
        grave_frame_count+=len(grave_frame_df)

        print(f"    --- {process_name:<20}• {cemetery_name:<20}• {cemetery_city:<20}• {cemetery_county:<20}• {cemetery_state:<10}• grave count: {grave_frame_count}/{memorial_count}")
        # print(grave_frame_df) 
        # input('grave_frame_df')
        sleep_time=random_sleep(frame_count)       
        time.sleep(sleep_time)
        
    end_time=datetime.datetime.now()
    run_processing_time=end_time-event_time
    event_time=end_time
    run_processing_time_seconds=int(run_processing_time.total_seconds())
    processing_time=end_time-start_time
    processing_time_minutes=int(processing_time.total_seconds())/60        
    print(f"*** {process_name:<20}• {cemetery_name:<20}• {cemetery_city:<20}• {cemetery_state:<10}• grave count: {grave_frame_count} •"+\
          f" cemetery count: {cemetery_count+1}/{total_cemetery_count} • cemetery id: {cemetery_id}• Processing Time: {run_processing_time_seconds} Sec • Cum Processing Time {processing_time_minutes:.1f} min")
    
graves_df['Siblings'] = graves_df['Siblings'].apply(lambda x: ' * '.join(x) if isinstance(x, list) else np.nan)
graves_df['Parents'] = graves_df['Parents'].apply(lambda x: ' * '.join(x) if isinstance(x, list) else np.nan)    

end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   

graves_df.to_csv(fileOut,mode="w", index=False,header=True)
graves_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)


**Starting ***  Extract Benjamin Graves   *** , Time: 2025-04-28 09:25:21.221817  **

    --- Extract Benjamin Graves • Mount Zion Cemetery • Somers              • Westchester County  • New York  • grave count: 100/449
    --- Extract Benjamin Graves • Mount Zion Cemetery • Somers              • Westchester County  • New York  • grave count: 200/449
    --- Extract Benjamin Graves • Mount Zion Cemetery • Somers              • Westchester County  • New York  • grave count: 300/449
    --- Extract Benjamin Graves • Mount Zion Cemetery • Somers              • Westchester County  • New York  • grave count: 400/449
    --- Extract Benjamin Graves • Mount Zion Cemetery • Somers              • Westchester County  • New York  • grave count: 449/449
*** Extract Benjamin Graves • Mount Zion Cemetery • Somers              • New York  • grave count: 449 • cemetery count: 1/22 • cemetery id: 2239418• Processing Time: 18 Sec • Cum Processing Time 0.3 min
    --- Extract Benjamin Graves • Mount Hebro

In [115]:
graves_df['Siblings'] = graves_df['Siblings'].apply(lambda x: ' • '.join(x) if isinstance(x, list) else np.nan)
graves_df['Parents'] = graves_df['Parents'].apply(lambda x: ' • '.join(x) if isinstance(x, list) else np.nan)    

end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   

graves_df.to_csv(fileOut,mode="w", index=False,header=True)
graves_df.to_csv(fileOut0,mode="w", index=False,header=True) 


**** Extract Benjamin Graves    Processing Time: 97 Sec


In [113]:


# A=graves_df['Siblings'].apply(lambda names: [name.replace(r',', '•') for name in names])
A=graves_df['Siblings'].apply(lambda names: [name.replace(',', '•') for name in names] if isinstance(names, list) else names)
A

graves_df['Siblings'][0:3]
# string_result = '[ ' + ' • '.join(graves_df['Siblings']) + ' ]'
# string_result = ' • '.join(graves_df['siblings']) if isinstance(graves_df['siblings'], list) else np.nan
# string_result



graves_df['Siblings'] = graves_df['Siblings'].apply(lambda x: ' • '.join(x) if isinstance(x, list) else np.nan)
B
graves_df['Parents'] = graves_df['Parents'].apply(lambda x: ' • '.join(x) if isinstance(x, list) else np.nan)
graves_df 

,cemeteryId,cemeteryName,fullName,lastName,firstName,birthDate,deathDate,Parents,Siblings
0,1084128,Verbena Cemetery,Claude Earnest Abernathy,Abernathy,Claude,5 Apr 1881,20 Nov 1971,NaN,NaN
1,1084128,Verbena Cemetery,Dewey Hobson Abernathy,Abernathy,Dewey,29 Sep 1908,15 Nov 1981,Claude Earnest Abernathy • Fannie Maebelle Cob...,Earnest Hobson Abernathy • Fannie Maebelle Cob...
2,1084128,Verbena Cemetery,Dorthula N. Dolly Cole Abernathy,Abernathy,Dorthula,5 Jun 1854,18 Oct 1933,Mrs Nancy Raspberry Vineyard Cole • David Laws...,David Lawson Cole • Mary Lou Cole Coggin • Nat...
3,1084128,Verbena Cemetery,Fannie Maebelle Cobern Abernathy,Abernathy,Fannie,7 Feb 1883,19 Jan 1946,Francis Flovenia Fannie Strock Cobern • George...,Edgar Charles Cobern • Elizabeth Lizzie Coburn...
4,1084128,Verbena Cemetery,Rev John Clarence Abernathy,Abernathy,John,23 Oct 1886,11 Dec 1918,Dorthula N. Dolly Cole Abernathy • John Thomas...,Alice Brown Brownie Abernathy Hodnett • Dollie...
...,...,...,...,...,...,...,...,...,...
95,1084128,Verbena Cemetery,Maude Mae Caffey Catts,Catts,Maude,17 Jun 1858,12 Aug 1938,Dr Hugh William Caffey • Jerusha May Rives Caffey,Dr Hugh William Caffey • Caroline Caffey Dudle...
96,1084128,Verbena Cemetery,Samuel Walker Catts,Catts,Samuel,18 Nov 1878,18 Nov 1960,John Smyly Catts • Maude Mae Caffey Catts,Gordon Rives Catts • Maude Mae Caffey Catts
97,1084128,Verbena Cemetery,Lewis E. Centery,Centery,Lewis,1899,1915,NaN,NaN
98,1084128,Verbena Cemetery,Oba Centery,Centery,Oba,1909,1910,NaN,NaN


In [ ]:
url
